In [2]:
import pandas as pd
import numpy as np
import string


# Read the data
X_full = pd.read_csv('./Data/email_origin.csv')
X_full
# X_full = X_full.head(1)

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.max_colwidth", None)
# pd.set_option("display.width", None)

# X_full.sample()
# print(X_full.loc[0,"text"])
# display(X_full)
# X_full.describe()
# X_full.info()

# for part in X_full.loc[0,"text"].walk():
#     print("Content type:", part.get_content_type())
#     print("Content maintype:", part.get_content_maintype())
#     print("Payload:")
#     print(part.get_payload(decode=True))
#     print("-------------------")

,label,origin
0,1,Return-Path: <RickyAmes@aol.com>\nReceived: fr...
1,0,Return-Path: <bounce-debian-mirrors=ktwarwic=s...
2,1,Return-Path: <7stocknews@tractionmarketing.com...
3,1,Return-Path: <vqucsmdfgvsg@ruraltek.com>\nRece...
4,1,Return-Path: <dcube@totalink.net>\nReceived: f...
...,...,...
75414,1,Return-Path: <Merrill8783@168city.com>\nReceiv...
75415,1,Return-Path: <Merrill8783@168city.com>\nReceiv...
75416,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...
75417,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...


In [3]:
# Data preprocessing 
import email
import re
import os

from tqdm import tqdm
from bs4 import BeautifulSoup

# Parse raw email string into structured email message object
def read_email_from_string(s):
    message = email.message_from_string(s)
    return message

# Extract actual email body from an email message object
def extract_email_body(message):
    # If message contains multipart like text/plain or text/html or image/gif
    if message.is_multipart():
        for part in message.walk():
            # text or image or etc
            type_content = part.get_content_maintype()
            # Get text part of content
            if type_content == 'text':
                message = part
                break
        else:
            return 'escapenonetext'

    # If text content is base64 encoded
    if message.get('Content-Transfer-Encoding') == 'base64':
        try:
            # Get payload in bytes and convert to string
            body = message.get_payload(decode=True).decode()
        except:
            # Try different decoding
            body = message.get_payload(decode=True).decode(encoding='ISO-8859-1')
    else:
        # no decoding
        body = message.get_payload(decode=False)
    return body

# Remove html tags 
def remove_html(s):
    # beautifulsoup object with the html tags
    soup = BeautifulSoup(s, 'lxml')
    for sp in soup(['script', 'style', 'head', 'meta', 'noscript']):
        # completely remove 
        sp.decompose()
    # join the remaining text strings 
    s = ' '.join(soup.stripped_strings)
    return s

# Text cleaning and normalization
def email_body_to_text(body):
    # Remove html tags
    body = remove_html(body)
    punctuation = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~'
    # remove punctuations
    body = re.sub('[{}]'.format(punctuation), ' ', body)
    # remove newline
    body = re.sub('\n+', ' ', body)
    # compress whitespace 
    body = re.sub('\\s+', ' ', body)
    # replace numbers
    body = re.sub(r'[0-9]+', 'escapenumber', body)
    body = body.lower()
    # replace unusually long strings
    body = re.sub(r'[a-z0-9]{20,}', 'escapelong', body)
    return body


In [4]:
X_full['text'] = X_full['origin'].map(read_email_from_string).map(extract_email_body).map(email_body_to_text)
X_full

/tmp/ipykernel_9401/2489748522.py:44: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(s, 'lxml')


,label,origin,text
0,1,Return-Path: <RickyAmes@aol.com>\nReceived: fr...,do you feel the pressure to perform and not ri...
1,0,Return-Path: <bounce-debian-mirrors=ktwarwic=s...,hi i've just updated from the gulus and i chec...
2,1,Return-Path: <7stocknews@tractionmarketing.com...,mega authenticv i a g r a discount pricec i a ...
3,1,Return-Path: <vqucsmdfgvsg@ruraltek.com>\nRece...,hey billy it was really fun going out the othe...
4,1,Return-Path: <dcube@totalink.net>\nReceived: f...,system of the home it will have the capabiliti...
...,...,...,...
75414,1,Return-Path: <Merrill8783@168city.com>\nReceiv...,while we may have high expectations of our ass...
75415,1,Return-Path: <Merrill8783@168city.com>\nReceiv...,while we may have high expectations of our ass...
75416,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,for those who are interested i just cook a lit...
75417,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,hello as i wrote i call sqlfetch channel t stu...


In [5]:
# Remove duplicates
X_full_drop_dup = X_full.drop_duplicates(subset='text')
X_full_drop_dup

,label,origin,text
0,1,Return-Path: <RickyAmes@aol.com>\nReceived: fr...,do you feel the pressure to perform and not ri...
1,0,Return-Path: <bounce-debian-mirrors=ktwarwic=s...,hi i've just updated from the gulus and i chec...
2,1,Return-Path: <7stocknews@tractionmarketing.com...,mega authenticv i a g r a discount pricec i a ...
3,1,Return-Path: <vqucsmdfgvsg@ruraltek.com>\nRece...,hey billy it was really fun going out the othe...
4,1,Return-Path: <dcube@totalink.net>\nReceived: f...,system of the home it will have the capabiliti...
...,...,...,...
75413,1,Return-Path: <hearst@cdrcorp.com>\nReceived: f...,versuchen sie unser produkt und sie werden fuh...
75414,1,Return-Path: <Merrill8783@168city.com>\nReceiv...,while we may have high expectations of our ass...
75416,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,for those who are interested i just cook a lit...
75417,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,hello as i wrote i call sqlfetch channel t stu...


In [6]:
# Check if there are any missing values or Nan values and remove them
X_full_na = X_full_drop_dup[~X_full_drop_dup['text'].str.contains(r"\b\w+\b", na=False)]
X_full_na


,label,origin,text
7,1,Return-Path: <dbmagyrln@dbmag.com>\nReceived: ...,


In [7]:
X_full_na = X_full_drop_dup[X_full_drop_dup['text'].str.contains(r"\b\w+\b", na=False)]
X_full_na

,label,origin,text
0,1,Return-Path: <RickyAmes@aol.com>\nReceived: fr...,do you feel the pressure to perform and not ri...
1,0,Return-Path: <bounce-debian-mirrors=ktwarwic=s...,hi i've just updated from the gulus and i chec...
2,1,Return-Path: <7stocknews@tractionmarketing.com...,mega authenticv i a g r a discount pricec i a ...
3,1,Return-Path: <vqucsmdfgvsg@ruraltek.com>\nRece...,hey billy it was really fun going out the othe...
4,1,Return-Path: <dcube@totalink.net>\nReceived: f...,system of the home it will have the capabiliti...
...,...,...,...
75413,1,Return-Path: <hearst@cdrcorp.com>\nReceived: f...,versuchen sie unser produkt und sie werden fuh...
75414,1,Return-Path: <Merrill8783@168city.com>\nReceiv...,while we may have high expectations of our ass...
75416,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,for those who are interested i just cook a lit...
75417,0,Return-Path: <r-help-bounces@stat.math.ethz.ch...,hello as i wrote i call sqlfetch channel t stu...


In [8]:
# Save 
X_full_na.to_csv("email_preprocessed.csv", index=False, escapechar='\\')